0. Load Libraries + Install API

In [1]:
#load libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import glob
import string
from tabulate import tabulate
import os
from pathlib import Path
import shutil
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString, box
from tqdm import tqdm
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
import math
import time
from shapely.ops import unary_union
import seaborn as sns
from scipy.spatial import cKDTree
import matplotlib.cm as cm
from matplotlib.patches import Circle
from collections import defaultdict
import pyomo.environ as pyo
import gurobipy as gp
import osmnx as ox, networkx as nx
from osmnx.distance import nearest_nodes
from descartes import PolygonPatch
import requests
import matplotlib.colors as colors
import pickle
from datetime import datetime
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

today = datetime.today().strftime("%Y%m%d")

1. Load Data

In [2]:
ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / 'intermediate_result'
RESULTS_DIR = ROOT_DIR / 'results'

In [3]:
# 오픈 소스 지도 데이터 (출처: OpenStreetMap, Live Update)
data_dir = ROOT_DIR / 'data' / "openstreetmap_region"
with open(ROOT_DIR / 'data' /"openstreetmap_region/G_한국.pkl", "rb") as f:
    G_한국 = pickle.load(f)
with open(ROOT_DIR / 'data' /"openstreetmap_region/G_울릉.pkl", "rb") as f:
    G_울릉 = pickle.load(f)
with open(ROOT_DIR / 'data' /"openstreetmap_region/G_제주.pkl", "rb") as f:
    G_제주 = pickle.load(f)

In [4]:
gdf_bogun = gpd.read_file(DATA_DIR / 'gdf_bogun.gpkg', encoding="utf-8-sig")
gdf_hosp = gpd.read_file(DATA_DIR / 'gdf_hosp.gpkg', encoding="utf-8-sig")
gdf_grid_pop = pd.read_csv(DATA_DIR / 'gdf_grid_pop.csv', encoding="utf-8-sig")
gdf_emd = pd.read_csv(DATA_DIR / 'gdf_emd.csv', encoding="utf-8-sig")

In [5]:
bogun_unique = gdf_bogun.loc[
    gdf_bogun.groupby("node_id")["offset_to_node_m"].idxmin(),
    ["node_id", "암호화요양기호"]
].reset_index(drop=True)

In [6]:
grid_nodes = gdf_grid_pop['node_id']

# 노드 → 해당 노드에 스냅된 격자 매핑
node_to_gridpos = defaultdict(list)
for pos, nid in enumerate(grid_nodes):
    node_to_gridpos[nid].append(pos)
node_to_gridpos = dict(node_to_gridpos)

# 행 위치 → 격자의 인덱스라벨 매핑
grid_index_labels = gdf_grid_pop.index.to_numpy()

snap_dist = gdf_grid_pop['offset_to_node_m']

In [7]:
# 각 보건소/보건지소별 20km 반경탐색
CUT_M = 20_000
rows = []
for _, row in bogun_unique.iterrows():
    src_node  = row["node_id"]
    bogun_id  = row["암호화요양기호"]

    # 기관 노드에서 시작하는 단일-소스 최단거리 (합쳐진 그래프 G 사용)
    if src_node in G_한국:
        dist_len = nx.single_source_dijkstra_path_length(
        G_한국, source=src_node, cutoff=CUT_M, weight="length"
        )
    elif src_node in G_울릉:
        dist_len = nx.single_source_dijkstra_path_length(
        G_울릉, source=src_node, cutoff=CUT_M, weight="length"
        )
    else:
        dist_len = nx.single_source_dijkstra_path_length(
        G_제주, source=src_node, cutoff=CUT_M, weight="length"
        )
    # 이 소스로부터 도달 가능한 모든 도로노드에 대해, 그 노드에 붙은 격자들을 수집
    for nid, road_m in dist_len.items():
        pos_list = node_to_gridpos.get(nid)
        if not pos_list:
            continue

        for pos in pos_list:
            # 총거리 = (노드→기관 도로망 거리) + (격자→노드 스냅 직선거리)
            total_km = road_m/1000.0 + (snap_dist[pos]/1000.0)

            rows.append({
                "grid_index": grid_index_labels[pos],
                "near_bogun_id": bogun_id,
                "near_bogun_dist_km": total_km
            })

In [8]:
# 각 grid에 영향을 주는 보건소/보건지소 (near_bogun_id)와 거리 (near_bogun_dist_km)
df_near = pd.DataFrame(rows)
df_near = (
    df_near
    .groupby(["grid_index", "near_bogun_id"], as_index=False)["near_bogun_dist_km"]
    .min()
)

In [9]:
from shapely import wkt

if isinstance(gdf_grid_pop.geometry.iloc[0], str):
    gdf_grid_pop["geometry"] = gdf_grid_pop["geometry"].apply(wkt.loads)


In [10]:
gdf_grid_pop = gpd.GeoDataFrame(
    gdf_grid_pop,
    geometry="geometry",
    crs="EPSG:5179"
)
gdf_grid_pop['grid_centroid'] = gdf_grid_pop.geometry.centroid

In [11]:
# 가장 가까운 종합병원/병원/의원/보건의료원과의 거리
# 결측치 제거
gdf_hosp = gdf_hosp[gdf_hosp.geometry.notna() & (~gdf_hosp.geometry.is_empty)].copy()
xy = np.c_[gdf_hosp.geometry.x.to_numpy(), gdf_hosp.geometry.y.to_numpy()]
mask = np.isfinite(xy[:,0]) & np.isfinite(xy[:,1])
gdf_hosp = gdf_hosp.loc[mask].reset_index(drop=True) 

# KDTree로 가장 가까운 종합병원/병원/의원/보건의료원 탐색
h_xy = np.c_[gdf_hosp.geometry.x.values, gdf_hosp.geometry.y.values]
tree = cKDTree(h_xy)
g_xy = np.c_[gdf_grid_pop['grid_centroid'].x.values, gdf_grid_pop['grid_centroid'].y.values]
dist_m, idx = tree.query(g_xy, k=1)

# 최종 거리 (nearest_hosp_dist_km) = 노드-> 기관 도로망상 거리 (dist_m) + 격자 중심 -> 노드 직선 거리 (snap_dist)
gdf_grid_pop["nearest_hosp_dist_km"] = dist_m/1000 + snap_dist/1000
gdf_grid_pop["nearest_hosp_idx"] = idx

# 기관 ID 추가
gdf_grid_pop["nearest_hosp_id"] = gdf_grid_pop["nearest_hosp_idx"].map(gdf_hosp[["요양기관명", "암호화요양기호"]]["암호화요양기호"])

# 불필요한 칼럼 제거
gdf_grid_pop = gdf_grid_pop.drop(columns = ['법정동코드', 'nearest_hosp_idx'])

C:\Users\hshoo\AppData\Local\Temp\ipykernel_17580\1261472619.py:3: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  gdf_hosp = gdf_hosp[gdf_hosp.geometry.notna() & (~gdf_hosp.geometry.is_empty)].copy()


4. 접근성 산출

In [12]:
# 함수 정의
def logistic_decay_5k(d, d0=2.5, k=0.5):
    return np.where(
        d <= 5,
        1.0 / (1.0 + np.exp((d - d0)/k)),
        0.0
    )
def logistic_decay_10k(d, d0=5, k=1):
    return np.where(
        d <= 10,
        1.0 / (1.0 + np.exp((d - d0)/k)),
        0.0
    )
def logistic_decay_20k(d, d0=10, k=2):
    return np.where(
        d <= 20,
        1.0 / (1.0 + np.exp((d - d0)/k)),
        0.0
    )

In [13]:
gdf_grid_pop["EMD_CD"] = (
    gdf_grid_pop["EMD_CD"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

gdf_emd["EMD_CD"] = (
    gdf_emd["EMD_CD"]
    .astype(str)
    .str.strip()
)

In [14]:
# 거리감쇠함수를 이용해 기존 의료기관 접근성 산출
gdf_grid_pop["accessibility_5k"] = gdf_grid_pop["nearest_hosp_dist_km"].apply(logistic_decay_5k)
gdf_grid_pop["accessibility_10k"] = gdf_grid_pop["nearest_hosp_dist_km"].apply(logistic_decay_10k)
gdf_grid_pop["accessibility_20k"] = gdf_grid_pop["nearest_hosp_dist_km"].apply(logistic_decay_20k)

# 읍면동코드로 동 or 읍면 매칭 시킴
gdf_emd['suffix'] = gdf_emd['EMD_KOR_NM'].str[-1]
eup_myeon_list = list(gdf_emd[gdf_emd['suffix'].isin(['읍', '면'])]['EMD_CD'])
dong_list = list(gdf_emd[gdf_emd['suffix'].isin(['동', '로', '가'])]['EMD_CD'])

# 격자별 접근성 구하기
gdf_grid_pop["EMD_CD"] = gdf_grid_pop["EMD_CD"].astype(str)
gdf_grid_pop["class"] = np.where(
    gdf_grid_pop["EMD_CD"].isin(dong_list), "동",
    np.where(gdf_grid_pop["EMD_CD"].isin(eup_myeon_list), "읍면", None)
)
gdf_grid_pop['accessibility_1'] = np.where(gdf_grid_pop['class'] == '동', gdf_grid_pop["accessibility_10k"], gdf_grid_pop["accessibility_20k"])
gdf_grid_pop['accessibility_2'] = np.where(gdf_grid_pop['class'] == '동', gdf_grid_pop["accessibility_5k"], gdf_grid_pop["accessibility_10k"])

# 인구가 살고 있는 격자의 accessibility_1과 accessibility_2 분포
gdf_grid_pop_inhab = gdf_grid_pop.loc[gdf_grid_pop['population'] > 0, ["class", "accessibility_1", "accessibility_2", "adjusted_population"]]

In [15]:
# df_near에 격자 속성 조인
dfN = df_near.merge(
    gdf_grid_pop_inhab[["class", "accessibility_1", "accessibility_2", "adjusted_population"]],
    left_on="grid_index", right_index=True,
    how="left"
)

# 거리감쇠 접근성 계산
dist = dfN["near_bogun_dist_km"].to_numpy()
pot5  = logistic_decay_5k(dist)
pot10 = logistic_decay_10k(dist)
pot20 = logistic_decay_20k(dist)

is_dong = (dfN["class"].to_numpy() == "동")
pot1 = np.where(is_dong, pot10, pot20)
pot2 = np.where(is_dong, pot5,  pot10)

dfN["bogun_accessibility_1"] = pot1
dfN["bogun_accessibility_2"] = pot2

# 기존 접근성 대비 증가분(delta >= 0)
a1 = dfN["accessibility_1"].to_numpy()
a2 = dfN["accessibility_2"].to_numpy()
dfN["delta_accessibility_1"] = np.maximum(0.0, pot1 - a1)
dfN["delta_accessibility_2"] = np.maximum(0.0, pot2 - a2)

# 수혜인구(각 (격자, 기관) 쌍의 기여치)
pop = pd.to_numeric(dfN["adjusted_population"], errors="coerce").fillna(0.0).to_numpy()
dfN["benefit_population_1"] = dfN["delta_accessibility_1"].to_numpy() * pop
dfN["benefit_population_2"] = dfN["delta_accessibility_2"].to_numpy() * pop

# 보건소/지소별 수혜인구 집계
benefit_by_bogun_1 = dfN.groupby("near_bogun_id", as_index=True)["benefit_population_1"].sum()
benefit_by_bogun_2 = dfN.groupby("near_bogun_id", as_index=True)["benefit_population_2"].sum()

# gdf_bogun에 매핑(기관 식별자는 암호화요양기호로 가정)
gdf_bogun = gdf_bogun.copy()
gdf_bogun["benefit_population_1"] = gdf_bogun["암호화요양기호"].map(benefit_by_bogun_1).fillna(0.0)
gdf_bogun["benefit_population_2"] = gdf_bogun["암호화요양기호"].map(benefit_by_bogun_2).fillna(0.0)

5. 최적화

In [16]:
# 거리감쇠 함수 기준 정하기
scenario = 'dong_10' # 동 : 10km, 읍면 : 20km
# scenario = 'dong_5' # 동 : 5km, 읍면 : 10km

if scenario == 'dong_10':
    delta_accessibility_scenario = 'delta_accessibility_1'
    accessibility_scenario = 'accessibility_1'
elif scenario == 'dong_5':
    delta_accessibility_scenario = 'delta_accessibility_2'
    accessibility_scenario = 'accessibility_2'

In [17]:
loc_bogun_filtered = pd.read_csv(DATA_DIR / 'loc_bogun_filtered.csv', encoding="utf-8-sig")

In [18]:
# 해당 dataframe을 기준으로 indexing 함
gdf_bogun = gdf_bogun.reset_index(drop=True).copy()
gdf_bogun["bogun_id"] = gdf_bogun.index.astype(str)
fac_to_instid = dict(zip(gdf_bogun["암호화요양기호"], gdf_bogun["bogun_id"]))

# 격자별 영향 받는 보건소/보건지소와 거리 정리 ()
df_pairs = dfN[["grid_index", "near_bogun_id", delta_accessibility_scenario]].copy()

# 노드ID -> 암호화요양기호 -> bogun_id 로 치환
df_pairs["bogun_id"] = df_pairs["near_bogun_id"].map(fac_to_instid)

# 숫자/결측 정리
df_pairs[delta_accessibility_scenario] = (
    pd.to_numeric(df_pairs[delta_accessibility_scenario], errors="coerce").fillna(0.0)
)

# 격자 - 보건소/보건지소 pair에서 해당 기관에 공보의를 배치하였을때 접근성이 변하는 정도
EPS = 1e-5
df_pairs = df_pairs[df_pairs["bogun_id"].notna() & (df_pairs[delta_accessibility_scenario] > EPS)]
df_pairs = df_pairs.groupby(["grid_index", "bogun_id"], as_index=False)[delta_accessibility_scenario].max()

# grid_to_inst, inst_to_grid 생성
df_pairs["grid_index"] = df_pairs["grid_index"].astype(str)
df_pairs["bogun_id"]   = df_pairs["bogun_id"].astype(str)

# 격자별 영향을 주는 보건소/보건지소 매핑 dict
grid_to_inst = defaultdict(dict)
for _, r in df_pairs.iterrows():
    i = r["grid_index"]; j = r["bogun_id"]; a = float(r[delta_accessibility_scenario])
    grid_to_inst[i][j] = a
grid_to_inst = dict(grid_to_inst)

# 보건소/보건지소별 영향을 주는 격자 매핑 dict
inst_to_grid = defaultdict(dict)
for _, r in df_pairs.iterrows():
    i = r["grid_index"]; j = r["bogun_id"]; a = float(r[delta_accessibility_scenario])
    inst_to_grid[j][i] = a
inst_to_grid = dict(inst_to_grid)

# (i,j)별 거리 테이블
_df_dist = dfN[["grid_index", "near_bogun_id", "near_bogun_dist_km"]].copy()
_df_dist["grid_index"] = _df_dist["grid_index"].astype(str)
_df_dist["bogun_id"]   = _df_dist["near_bogun_id"].map(fac_to_instid).astype(str)
_df_dist = _df_dist.dropna(subset=["bogun_id"])

# J_i^*: 격자 i의 후보 시설 집합
Jstar = {}
for i, g in _df_dist.groupby("grid_index"):
    Jstar[i] = frozenset(g["bogun_id"].unique().tolist())

# d_ij: 거리 dict
d_ij = {(row.grid_index, row.bogun_id): float(row.near_bogun_dist_km)
        for row in _df_dist.itertuples(index=False)}

# C_ij: j보다 더 가까운 시설들의 집합
Cij = {}
for (i, j), dij in d_ij.items():
    # 동일 격자 i의 다른 후보들 중 j보다 가까운 c 수집
    Cij[(i, j)] = frozenset(
        c for c in Jstar.get(i, [])
        if c != j and d_ij.get((i, c), float("inf")) < dij
    )

# 현재 배치안 기준 시군구별 총 배치 공보의 수
loc_bogun_filtered['시군구코드'] = loc_bogun_filtered["EMD_CD"].astype(str).str[:5]
allocated = loc_bogun_filtered[loc_bogun_filtered["현재공보의배치여부"].astype(float) == 1.0]
target_sgg_series = allocated.groupby("시군구코드").size()
target_sgg = target_sgg_series.to_dict()

# 기관 -> 시군구 매핑
gdf_bogun_sgg = pd.merge(gdf_bogun, loc_bogun_filtered[['암호화요양기호', '시군구코드']], on = '암호화요양기호', how = 'inner')
inst_to_sgg = (
    gdf_bogun_sgg.dropna(subset=["bogun_id", "시군구코드"])
             .drop_duplicates(subset=["bogun_id"])
             .set_index("bogun_id")["시군구코드"]
             .to_dict()
)

# 이미 배치가 되어 있는 기관은 inst_to_grid가 비어있더라도 후보로 추가
alloc_inst_ids = (
    allocated["암호화요양기호"]
    .map(fac_to_instid)
    .dropna()
    .astype(str)
    .unique()
)
for j in alloc_inst_ids:
    if j not in inst_to_grid:
        inst_to_grid[j] = {}

print('남아 있는 격자는 인구가 1명 이상이며, 배치 결과에 따라 접근성 증가가 있을 수 있는 격자들임')
print(f"격자 수(|I|): {len(grid_to_inst)}")
print(f"기관 수(|J|): {len(inst_to_grid)}")

남아 있는 격자는 인구가 1명 이상이며, 배치 결과에 따라 접근성 증가가 있을 수 있는 격자들임
격자 수(|I|): 58601
기관 수(|J|): 1322


In [19]:
# 현재 배치안 기준 시군구별 총 배치 공보의 수
loc_bogun_filtered['시군구코드'] = loc_bogun_filtered["EMD_CD"].astype(str).str[:5]
allocated_0 = loc_bogun_filtered[loc_bogun_filtered["2026년 전역"].astype(int) == 1.0]
allocated_1 = loc_bogun_filtered[loc_bogun_filtered["2027년 전역"].astype(int) == 1.0]
allocated_2 = loc_bogun_filtered[loc_bogun_filtered["2028년 전역"].astype(int) == 1.0]

# timestep dynamics 용
target_sgg_series_0 = allocated_0.groupby("시군구코드").size()
target_sgg_series_1 = allocated_1.groupby("시군구코드").size()
target_sgg_series_2 = allocated_2.groupby("시군구코드").size()
target_sgg_d = {}
target_sgg_d[0] = target_sgg_series_0.to_dict()
target_sgg_d[1] = target_sgg_series_1.to_dict()
target_sgg_d[2] = target_sgg_series_2.to_dict()

In [20]:
# 연도별 공보의 배치안
end_2026 = gdf_bogun['2026년 전역']
end_2027 = gdf_bogun['2027년 전역']
end_2028 = gdf_bogun['2028년 전역']
end_2026 = dict(end_2026)
end_2027 = dict(end_2027)
end_2028 = dict(end_2028)
end_2026 = {k : v for k, v in end_2026.items() if str(k) in inst_to_grid}
end_2027 = {k : v for k, v in end_2027.items() if str(k) in inst_to_grid}
end_2028 = {k : v for k, v in end_2028.items() if str(k) in inst_to_grid}

# 문자열로 변환
end_2026 = {str(k): v for k, v in end_2026.items()}
end_2027 = {str(k): v for k, v in end_2027.items()}
end_2028 = {str(k): v for k, v in end_2028.items()}

In [ ]:
# MODE = 'free' -> 무제약
# MODE = 'sgg_relaxed' -> 일정증감
# MODE = 'sgg_fixed' -> 현재유지

# version = static -> 기존 static solution
# version = dynamics -> dynamics solution 

# 사전 계산
param_grid = [
    # {"N": {0:293, 1:172, 2:88, 3:80, 4:75, 5:70, 6:65, 7:60}, "LAMBDA_": 0.5, "T": 0.5, "MODE": "sgg_fixed", "t": 5, "start" : 3, 'version' : 'dynamics'}
    {"N": 195, "LAMBDA_": 0.5, "T": 0.5, "MODE": "free", 'version' : 'static'}
]

# 일정증감 시나리오에서 얼마만큼 +-이 허용되는지
SLACK = 2

# 결과 저장 경로
output_dir = RESULTS_DIR / "optimization_results"
os.makedirs(output_dir, exist_ok=True)

results_summary = []

for param in param_grid:
    version  = param['version']
    # dynamics version
    if version == 'dynamics':
        N        = param["N"]
        LAMBDA_  = param["LAMBDA_"]
        T        = param["T"]
        MODE     = param["MODE"]
        t        = param["t"]
        start    = param["start"]

        print(f"\n=== Running optimization for N={N}, λ={LAMBDA_}, T={T}, MODE={MODE}, timestep={t}, start={start} ===")

        # --------------------------
        # PRE-PROCESSING
        # --------------------------
        I = sorted(grid_to_inst.keys())
        J = sorted(inst_to_grid.keys())

        gdf_grid_pop_stridx = gdf_grid_pop.copy()
        gdf_grid_pop_stridx.index = gdf_grid_pop_stridx.index.astype(str)

        # 접근성 a_i (시나리오 컬럼 사용)
        a_dict = (
            gdf_grid_pop_stridx
            .loc[gdf_grid_pop_stridx.index.isin(list(grid_to_inst.keys())), accessibility_scenario]
            .clip(0, 1)
            .to_dict()
        )

        # timestep
        timestep = range(0, t + 3)

        # 취약격자 K
        K = [i for i in I if a_dict[i] <= T]
        print(f'기존 접근성이 {T} 이하인 취약 격자 수 : {len(K)}')

        # rolling window 인덱스 미리 만들기
        ROLL = {
            t: [tau for tau in timestep if max(0, t-2) <= tau <= t]
            for t in timestep
        }

        # 파라미터 사전
        b_dict = {(grid, inst): delta_a
                for grid, inst_dict in grid_to_inst.items()
                for inst, delta_a in inst_dict.items()}
        w_dict = gdf_grid_pop_stridx.loc[gdf_grid_pop_stridx.index.isin(I), "adjusted_population"].to_dict()
        Ji     = {i: frozenset(j_delta.keys()) for i, j_delta in grid_to_inst.items()}
        q_dict = {i: math.sqrt(a_dict[i]) for i in I}

        # 정규화 상수(원래 정의 유지)
        ALPHA = sum(1.0 - q_dict[k] for k in K)
        BETA  = sum(w_dict[i]*(1.0 - q_dict[i]) for i in I)
        print(f'ALPHA : {ALPHA:.4f}, BETA : {BETA:.4f}')

        # 탐욕 초기해(웜스타트)
        inst_score  = {j: sum(v.values()) for j, v in inst_to_grid.items()}
        greedy_topk = {}
        active_until = {j: -1 for j in inst_score} # j가 활성화된 경우 언제까지 활성화인지
        for time in timestep:
            candidates = [j for j in inst_score if active_until[j] < time]
            ranked = sorted(candidates, key=lambda j : inst_score[j], reverse = True)
            greedy_topk[time] = ranked[:N[time]]
            for j in greedy_topk[time]:
                active_until[j] = time + 2

        # 클리핑된 b_ij (a_i<1.0인 경우만)
        b_nz = {(i, j): min(v, 1.0 - a_dict[i])
                for i, sub in grid_to_inst.items() if a_dict[i] < 1.0
                for j, v in sub.items()}
        
        # 사전에 계산해놓기
        delta_gain = {
            (i, j): math.sqrt(a_dict[i] + b_nz[i, j]) - math.sqrt(a_dict[i])
            for (i, j) in b_nz
        }

        # --------------------------
        # MODEL
        # --------------------------
        m = pyo.ConcreteModel()
        m.I, m.K, m.J, m.t = pyo.Set(initialize=I), pyo.Set(initialize=K), pyo.Set(initialize=J), pyo.Set(initialize=timestep, ordered=True)

        m.Ji = pyo.Set(m.I, initialize=lambda m,i: Ji.get(i, set()), within=m.J)

        # 시간에 따른 sgg_fixed, sgg_relax 구현
        if MODE in {"sgg_fixed", "sgg_relaxed"}:
            all_sgg = sorted(set().union(*[d.keys() for d in target_sgg_d.values()]))
            Js = {sgg: [j for j in J if inst_to_sgg.get(j) == sgg] for sgg in all_sgg}
            m.S  = pyo.Set(initialize=sorted(set().union(*[d.keys() for d in target_sgg_d.values()])))
            m.Js = pyo.Set(m.S, initialize=lambda m,s: Js.get(s, []), within=m.J)
            m.target = pyo.Param(
                m.S, m.t,
                initialize=lambda m, s, t: target_sgg_d.get(t, {}).get(s, 0),
                within=pyo.NonNegativeIntegers
            )
            if MODE == "sgg_relaxed":
                m.slack = pyo.Param(initialize=SLACK)

        m.lmbd = pyo.Param(initialize=LAMBDA_)
        m.w = pyo.Param(m.I, initialize=w_dict, within=pyo.NonNegativeReals)
        m.a = pyo.Param(m.I, initialize=a_dict, within=pyo.UnitInterval)
        m.b = pyo.Param(m.I, m.J, initialize=b_nz, default=0.0, within=pyo.NonNegativeReals)
        m.q = pyo.Param(m.I, initialize=q_dict)
        m.N = pyo.Param(
            m.t,
            initialize=lambda m, t: N[t],
            within=pyo.NonNegativeIntegers
        )
        m.alpha, m.beta = pyo.Param(initialize=ALPHA), pyo.Param(initialize=BETA)
        m.E = pyo.Set(dimen=2, initialize=list(b_nz.keys()))
        m.delta = pyo.Param(
            m.E,
            initialize=delta_gain,
            default=0.0,
            within=pyo.NonNegativeReals
        )

        # Decision variables
        m.n = pyo.Var(m.J, m.t, domain=pyo.Binary) # location j에 공보의 신규 배치 여부를 binary variable로 결정
        m.y = pyo.Var(m.E, m.t, domain=pyo.Binary)

        # --------------------------
        # CONSTRAINTS
        # --------------------------
        # 공보의 수
        # timestep t에 신규 배치되는 총 공보의 수 Σ_j n_jt ≤ 1, ∀t
        def budget_rule(m, t):
            return sum(m.n[j, t] for j in m.J) == m.N[t]
        m.budget = pyo.Constraint(m.t, rule=budget_rule)
        
        # y_ij가 1이면 j에 배치되어야 함: y_ijt ≤ x_jt
        def link_y_n(m, i, j, t):
            return m.y[i, j, t] <= sum(
            m.n[j, tau] for tau in ROLL[t]
        )
        m.link_yx = pyo.Constraint(m.E, m.t, rule=link_y_n)

        # j보다 가까운 설치시설이 있으면 y_ijt = 0 : y_ijt ≤ 1 - x_ct, ∀c ∈ C_ij, ∀t
        def closest_rule(m, i, j, c, t):
            return m.y[i, j, t] <= 1 - sum(
                m.n[c, tau] for tau in m.t if t-2 <= tau <= t
            )
        m.closest = pyo.Constraint(
            ((i, j, c, t)
            for (i, j) in m.E
            for c in Cij.get((i, j), ())   # C_ij
            for t in m.t),
            rule=closest_rule
        )

        # 격자별 최대 하나의 기관만 고려함: Σ_j y_ijt ≤ 1, ∀i, ∀t
        def at_most_one(m, i, t):
            if len(m.Ji[i]) == 0:
                return pyo.Constraint.Skip
            return sum(m.y[i, j, t] for j in m.Ji[i] if (i, j) in m.E) <= 1
        m.at_most_one = pyo.Constraint(m.I, m.t, rule=at_most_one)

        # 공보의의 복무 기간은 3년이며 한 보건소에는 3년 동안 최대 1명의 공보의가 존재한다고 가정 -> 기존 x_jt 대체
        def rolling_limit_rule(m, j, t):
            return (sum(m.n[j, tau] for tau in ROLL[t]) <= 1)
        m.rolling_limit = pyo.Constraint(m.J, m.t, rule=rolling_limit_rule)

        # 배치안을 현재 배치안으로 고정 
        def existing_fixed_0(m, j): 
            if j not in end_2026: 
                return (m.n[j,0] == 0)
            return (m.n[j,0] == (1 if end_2026[j] else 0)) 
        def existing_fixed_1(m, j): 
            if j not in end_2027: 
                return (m.n[j,1] == 0)
            return (m.n[j,1] == (1 if end_2027[j] else 0)) 
        def existing_fixed_2(m, j): 
            if j not in end_2028: 
                return (m.n[j,2] == 0)
            return (m.n[j,2] == (1 if end_2028[j] else 0)) 
        if start >= 3: 
            m.existing_fixed_0 = pyo.Constraint(m.J, rule = existing_fixed_0) 
            m.existing_fixed_1 = pyo.Constraint(m.J, rule = existing_fixed_1) 
            m.existing_fixed_2 = pyo.Constraint(m.J, rule = existing_fixed_2) 
        elif start == 2: 
            m.existing_fixed_0 = pyo.Constraint(m.J, rule = existing_fixed_0) 
            m.existing_fixed_1 = pyo.Constraint(m.J, rule = existing_fixed_1) 
        elif start == 1: 
            m.existing_fixed_0 = pyo.Constraint(m.J, rule = existing_fixed_0)

        # 현재 배치안 대비 시군구 제약
        if start < 3:
            if MODE == "sgg_fixed":
                def sgg_fix(m, s, t):
                    if t > 2:
                        return pyo.Constraint.Skip
                    if len(m.Js[s]) == 0:
                        return pyo.Constraint.Skip
                    return sum(m.n[j, t] for j in m.Js[s]) == m.target[s, t]
                m.sgg_fix = pyo.Constraint(m.S, m.t, rule=sgg_fix)
            elif MODE == "sgg_relaxed":
                def sgg_lower(m, s, t):
                    if t > 2:
                        return pyo.Constraint.Skip
                    if len(m.Js[s]) == 0:
                        return pyo.Constraint.Skip
                    lb = max(int(m.target[s, t]) - int(m.slack), 0)
                    return sum(m.n[j, t] for j in m.Js[s]) >= lb
                def sgg_upper(m, s, t):
                    if t > 2:
                        return pyo.Constraint.Skip
                    if len(m.Js[s]) == 0:
                        return pyo.Constraint.Skip
                    ub = min(len(m.Js[s]), int(m.target[s, t]) + int(m.slack))
                    return sum(m.n[j, t] for j in m.Js[s]) <= ub
                m.sgg_lower = pyo.Constraint(m.S, m.t, rule=sgg_lower)
                m.sgg_upper = pyo.Constraint(m.S, m.t, rule=sgg_upper)

        # Objective
        m.obj = pyo.Objective(
            expr=m.lmbd   * (1/m.alpha) * sum(m.delta[k, j] * m.y[k, j, t] for (k, j) in m.E for t in m.t if k in m.K)
            + (1-m.lmbd) * (1/m.beta)  * sum(m.w[i]*(m.delta[i, j] * m.y[i, j, t]) for (i, j) in m.E for t in m.t),
            sense=pyo.maximize
        )

        # Warm start
        for j in m.J:
            for t in m.t:
                m.n[j, t].value = 1 if j in greedy_topk.get(t,[]) else 0

        # Solve
        solver = pyo.SolverFactory("gurobi")
        solver.options.update({"TimeLimit": 3600, "MIPGap": 0.01, "Threads": 2})
        results = solver.solve(m, tee=True)

        # --------------------------
        # RESULTS
        # --------------------------
        n_sol = pd.Series({(j, t) : pyo.value(m.n[j, t]) for j in m.J for t in m.t}, name="selected")
        selected_facilities = {}
        for t in m.t:
            selected_facilities[t] = {j for j in m.J if n_sol[j,t] > 0.5}
    
        # x_sol reconstruct
        x_sol = pd.Series({(j, t) : 1 if sum(n_sol[j, tau] for tau in ROLL[t]) > 0.5 else 0 for j in m.J for t in m.t}, name='activated')
        activated_facilities = {}
        for t in m.t:
            activated_facilities[t] = {j for j in m.J if x_sol[j, t] > 0.5}

        fairness_term = {}
        efficiency_term = {}
        obj_val         = pyo.value(m.obj)

        # 선택 결과 저장
        tag = {"free":"free", "sgg_fixed":"sgg_fixed", "sgg_relaxed":"sgg_relaxed"}[MODE]
        sel_path = output_dir / f"{tag}_timestep{t}_start{start}_lambda{LAMBDA_}_T{T}_{today}.pkl"

        result_dict = {
            "selected_facilities" : selected_facilities,
            "activated_facilities" : activated_facilities
        }
        with open(sel_path, "wb") as f:
            pickle.dump(result_dict, f)

        # 취약격자 중 개선 발생 격자 수
        improved_grids = {}
        for t in timestep:
            selected_set = selected_facilities[t]
            improved_grids[t] = []
            for grid_id in K:
                inst_dict = grid_to_inst.get(grid_id, {})
                if any(inst_id in selected_set for inst_id in inst_dict):
                    improved_grids[t].append(grid_id)

        for t in m.t:
            if t < start:
                continue
            print(f"총 취약 격자 수: {len(K)}")
            print(f"time {t} 배치 후 접근성 상승이 발생한 취약 격자 수: {len(improved_grids[t])}")

    # static version
    elif version == 'static':
        N        = param["N"]
        LAMBDA_  = param["LAMBDA_"]
        T        = param["T"]
        MODE     = param["MODE"]

        print(f"\n=== Running optimization for N={N}, λ={LAMBDA_}, T={T}, MODE={MODE} ===")

        # --------------------------
        # PRE-PROCESSING
        # --------------------------
        I = sorted(grid_to_inst.keys())
        J = sorted(inst_to_grid.keys())

        gdf_grid_pop_stridx = gdf_grid_pop.copy()
        gdf_grid_pop_stridx.index = gdf_grid_pop_stridx.index.astype(str)

        # 접근성 a_i (시나리오 컬럼 사용)
        a_dict = (
            gdf_grid_pop_stridx
            .loc[gdf_grid_pop_stridx.index.isin(list(grid_to_inst.keys())), accessibility_scenario]
            .clip(0, 1)
            .to_dict()
        )

        # 취약격자 K
        K = [i for i in I if a_dict[i] <= T]
        print(f'기존 접근성이 {T} 이하인 취약 격자 수 : {len(K)}')

        # 파라미터 사전
        b_dict = {(grid, inst): delta_a
                for grid, inst_dict in grid_to_inst.items()
                for inst, delta_a in inst_dict.items()}
        w_dict = gdf_grid_pop_stridx.loc[gdf_grid_pop_stridx.index.isin(I), "adjusted_population"].to_dict()
        Ji     = {i: frozenset(j_delta.keys()) for i, j_delta in grid_to_inst.items()}
        q_dict = {i: math.sqrt(a_dict[i]) for i in I}

        # 정규화 상수(원래 정의 유지)
        ALPHA = sum(1.0 - q_dict[k] for k in K)
        BETA  = sum(w_dict[i]*(1.0 - q_dict[i]) for i in I)
        print(f'ALPHA : {ALPHA:.4f}, BETA : {BETA:.4f}')

        # 탐욕 초기해(웜스타트)
        inst_score  = {j: sum(v.values()) for j, v in inst_to_grid.items()}
        greedy_topk = sorted(inst_score, key=inst_score.get, reverse=True)[:N]

        # 클리핑된 b_ij (a_i<1.0인 경우만)
        b_nz = {(i, j): min(v, 1.0 - a_dict[i])
                for i, sub in grid_to_inst.items() if a_dict[i] < 1.0
                for j, v in sub.items()}

        # --------------------------
        # MODEL
        # --------------------------
        m = pyo.ConcreteModel()
        m.I, m.K, m.J = pyo.Set(initialize=I), pyo.Set(initialize=K), pyo.Set(initialize=J)
        m.Ji = pyo.Set(m.I, initialize=lambda m,i: Ji.get(i, set()), within=m.J)

        if MODE in {"sgg_fixed", "sgg_relaxed"}:
            Js = {sgg: [j for j in J if inst_to_sgg.get(j) == sgg] for sgg in target_sgg.keys()}
            m.S  = pyo.Set(initialize=sorted(target_sgg.keys()))
            m.Js = pyo.Set(m.S, initialize=lambda m,s: Js.get(s, []), within=m.J)
            m.target = pyo.Param(m.S, initialize=target_sgg, within=pyo.NonNegativeIntegers)
            if MODE == "sgg_relaxed":
                m.slack = pyo.Param(initialize=SLACK)

        m.n, m.lmbd = pyo.Param(initialize=N), pyo.Param(initialize=LAMBDA_)
        m.w = pyo.Param(m.I, initialize=w_dict, within=pyo.NonNegativeReals)
        m.a = pyo.Param(m.I, initialize=a_dict, within=pyo.UnitInterval)
        m.b = pyo.Param(m.I, m.J, initialize=b_nz, default=0.0, within=pyo.NonNegativeReals)
        m.q = pyo.Param(m.I, initialize=q_dict)
        m.alpha, m.beta = pyo.Param(initialize=ALPHA), pyo.Param(initialize=BETA)

        # Decision variables
        m.x = pyo.Var(m.J, domain=pyo.Binary)
        m.E = pyo.Set(dimen=2, initialize=list(b_nz.keys()))
        m.y = pyo.Var(m.E, domain=pyo.Binary)
        m.z = pyo.Var(m.I, domain=pyo.NonNegativeReals)
        m.p = pyo.Var(m.I, bounds=lambda m,i: (q_dict[i], 1.0))

        # --------------------------
        # CONSTRAINTS
        # --------------------------
        # 공보의 수
        m.budget = pyo.Constraint(expr=sum(m.x[j] for j in m.J) == m.n)

        # y_ij가 1이면 j에 배치되어야 함: y_ij ≤ x_j
        def link_yx(m, i, j):
            return m.y[i, j] <= m.x[j]
        m.link_yx = pyo.Constraint(m.E, rule=link_yx)

        # j보다 가까운 설치시설이 있으면 y_ij = 0 : y_ij ≤ 1 - x_c, ∀c ∈ C_ij
        m.closest = pyo.ConstraintList()
        for (i, j) in m.E:
            for c in Cij.get((i, j), ()):
                m.closest.add(m.y[i, j] <= 1 - m.x[c])

        # 격자별 최대 하나의 기관만 고려함: Σ_j y_ij ≤ 1
        def at_most_one(m, i):
            if len(m.Ji[i]) == 0:
                return pyo.Constraint.Skip
            return sum(m.y[i, j] for j in m.Ji[i] if (i, j) in m.E) <= 1
        m.at_most_one = pyo.Constraint(m.I, rule=at_most_one)

        # z_i 정의: z_i = Σ_j b_ij * y_ij
        def z_def(m, i):
            return m.z[i] == sum(m.b[i, j] * m.y[i, j] for j in m.Ji[i] if (i, j) in m.E)
        m.z_def = pyo.Constraint(m.I, rule=z_def)

        # z의 upper bound -> redundant한 constraint인데 혹시 몰라 일단 추가함
        def z_cap(m, i):
            return m.z[i] <= 1.0 - m.a[i]
        m.z_cap = pyo.Constraint(m.I, rule=z_cap)

        # p_i = sqrt(a_i + z_i)
        m.define_p = pyo.Constraint(m.I, rule=lambda m,i: m.p[i]**2 == m.a[i] + m.z[i])

        # 현재 배치안 대비 시군구 제약
        if MODE == "sgg_fixed":
            def sgg_fix(m, s):
                if len(m.Js[s]) == 0:
                    return pyo.Constraint.Skip
                return sum(m.x[j] for j in m.Js[s]) == m.target[s]
            m.sgg_fix = pyo.Constraint(m.S, rule=sgg_fix)
        elif MODE == "sgg_relaxed":
            def sgg_lower(m, s):
                if len(m.Js[s]) == 0:
                    return pyo.Constraint.Skip
                lb = max(int(m.target[s]) - int(m.slack), 0)
                return sum(m.x[j] for j in m.Js[s]) >= lb
            def sgg_upper(m, s):
                if len(m.Js[s]) == 0:
                    return pyo.Constraint.Skip
                ub = min(len(m.Js[s]), int(m.target[s]) + int(m.slack))
                return sum(m.x[j] for j in m.Js[s]) <= ub
            m.sgg_lower = pyo.Constraint(m.S, rule=sgg_lower)
            m.sgg_upper = pyo.Constraint(m.S, rule=sgg_upper)

        # Objective
        m.obj = pyo.Objective(
            expr=m.lmbd   * (1/m.alpha) * sum(m.p[k] - m.q[k] for k in m.K)
            + (1-m.lmbd) * (1/m.beta)  * sum(m.w[i]*(m.p[i] - m.q[i]) for i in m.I),
            sense=pyo.maximize
        )

        # Warm start
        for j in m.J:
            m.x[j].value = 1 if j in greedy_topk else 0

        # Solve
        solver = pyo.SolverFactory("gurobi")
        solver.options.update({"TimeLimit": 3600, "MIPGap": 0.005, "Threads": 0})
        results = solver.solve(m, tee=True)

        # --------------------------
        # RESULTS
        # --------------------------
        x_sol = pd.Series({j: pyo.value(m.x[j]) for j in m.J}, name="selected")
        selected_facilities = [j for j in m.J if x_sol[j] > 0.5]

        obj_val         = pyo.value(m.obj)
        fairness_term   = pyo.value((1/m.alpha)*sum(m.p[k]-m.q[k] for k in m.K))
        efficiency_term = pyo.value((1/m.beta) *sum(m.w[i]*(m.p[i]-m.q[i]) for i in m.I))

        # 선택 결과 저장
        tag = {"free":"free", "sgg_fixed":"sgg_fixed", "sgg_relaxed":"sgg_relaxed"}[MODE]
        sel_path = output_dir / f"{tag}_N{N}_lambda{LAMBDA_}_T{T}_{today}.pkl"
        with open(sel_path, "wb") as f:
            pickle.dump(selected_facilities, f)

        # 취약격자 중 개선 발생 격자 수
        selected_set = set(selected_facilities)
        improved_grids = []
        for grid_id in K:
            inst_dict = grid_to_inst.get(grid_id, {})
            if any(inst_id in selected_set for inst_id in inst_dict.keys()):
                improved_grids.append(grid_id)

        print(f"총 취약 격자 수: {len(K)}")
        print(f"배치 후 접근성 상승이 발생한 취약 격자 수: {len(improved_grids)}")


# 요약 저장
print("\nAll runs completed")


=== Running optimization for N=195, λ=0.5, T=0.5, MODE=free ===
기존 접근성이 0.5 이하인 취약 격자 수 : 10065
ALPHA : 5400.9799, BETA : 241437.8367
Read LP format model from file C:\Users\hshoo\AppData\Local\Temp\tmp9kuf4o2e.pyomo.lp
Reading time = 0.80 seconds
x1: 277639 rows, 195920 columns, 476984 nonzeros
Set parameter TimeLimit to value 3600
Set parameter MIPGap to value 0.005
Set parameter Threads to value 0
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-1340P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  3600
MIPGap  0.005

Optimize a model with 277639 rows, 195920 columns and 476984 nonzeros (Max)
Model fingerprint: 0xc5252f9d
Model has 58602 linear objective coefficients
Model has 58601 quadratic constraints
Variable types: 117203 continuous, 78717 integer (78717 binary)
Coefficient statistics:
  Matrix range    

ValueError: No value for uninitialized VarData object p['10024']